# 00 — Thiết lập môi trường
Cài thư viện, mount Drive, kiểm tra API key trong Colab Secrets, và kiểm tra tài nguyên riêng cho PySCF (CPU/RAM) và cho JAX (GPU).

In [ ]:
# Cài thư viện (chỉ chạy trên Colab; runtime Colab bị reset sau mỗi phiên nên mỗi notebook tự cài thứ nó cần)
import sys
if "google.colab" in sys.modules:
    %pip install -q pyscf pymatgen mp-api h5py optax matplotlib

In [ ]:
import os, sys, json, time
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/ml_xc_colab"
else:
    ROOT = os.environ.get("ML_XC_ROOT", os.getcwd())
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)          # để import được thư mục utils/
DATA, RAW, PROC = f"{ROOT}/data", f"{ROOT}/data/raw", f"{ROOT}/data/processed"
print("ROOT =", ROOT)

## 1. Tạo cây thư mục và kiểm tra `utils/`

In [ ]:
for d in ["raw/cif", "processed", "densities", "reference_labels", "models"]:
    os.makedirs(f"{DATA}/{d}", exist_ok=True)

need = ["mp_api_utils", "structure_utils", "dft_utils", "density_utils", "data_utils", "model_utils"]
missing = [m for m in need if not os.path.exists(f"{ROOT}/utils/{m}.py")]
assert not missing, f"Thiếu file trong {ROOT}/utils/: {missing}"
print("utils/ đầy đủ:", need)

## 2. API key Materials Project
Key phải nằm trong **Colab Secrets** với tên `MP_API_KEY` (biểu tượng chìa khóa ở thanh bên trái, bật *Notebook access*). Không dán giá trị key vào code hay vào tên secret. Cell này chỉ kiểm tra có key hay không, không in key.

In [ ]:
try:
    from google.colab import userdata
    has_key = bool(userdata.get("MP_API_KEY"))
except Exception:
    has_key = bool(os.environ.get("MP_API_KEY"))
print("MP_API_KEY:", "có" if has_key else "CHƯA có — tạo secret MP_API_KEY trước khi chạy 01")

## 3. Tài nguyên cho PySCF
PySCF PBC trong pipeline này chạy trên **CPU** (OpenMP); GPU chỉ dùng cho huấn luyện JAX ở bước 07. Benchmark nhỏ bên dưới cho biết tốc độ thực của runtime hiện tại.

In [ ]:
from utils import dft_utils as dft
res = dft.resources()
print(json.dumps(res, indent=2, ensure_ascii=False))

In [ ]:
# Benchmark: Al fcc, basis nhỏ, k-mesh thô (thường vài giây đến vài chục giây)
from pymatgen.core import Structure, Lattice
al = Structure.from_spacegroup("Fm-3m", Lattice.cubic(4.05), ["Al"], [[0, 0, 0]]).get_primitive_structure()
t0 = time.time()
info, rho, dm, kpts, cell = dft.run_scf(al, 0.0, dict(basis="gth-szv-molopt-sr", precision=1e-6, kspacing=0.9))
bench = {"wall_s": round(time.time() - t0, 1), "nk": info["nk"], "cycles": info["cycles"],
         "converged": info["converged"], "electron_ok": info["electron_ok"],
         "s_per_k_per_cycle": round((time.time() - t0) / max(info["nk"] * max(info["cycles"], 1), 1), 3)}
print(bench)

## 4. JAX (huấn luyện)

In [ ]:
import jax
devs = jax.devices()
print("JAX devices:", devs)
if all(d.platform == "cpu" for d in devs):
    print("Không có GPU: vẫn huấn luyện được nhưng chậm hơn. Runtime > Change runtime type > GPU nếu cần ở bước 07.")

In [ ]:
report = {"resources": res, "pyscf_benchmark": bench, "jax_devices": [str(d) for d in devs],
          "mp_api_key_present": has_key}
json.dump(report, open(f"{PROC}/env_report.json", "w"), indent=2, ensure_ascii=False)
print("Đã lưu", f"{PROC}/env_report.json")